# 01 — Umami Analytics Ingestion

Validate the Umami Cloud REST API integration:
- Connect with API key
- Fetch website stats, pageviews, metrics, events
- Parse into our `Insights` model
- Save to local state (mock S3)

In [1]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os

load_dotenv('../.env')

UMAMI_API_KEY = os.getenv('UMAMI_API_KEY')
UMAMI_WEBSITE_ID = os.getenv('UMAMI_WEBSITE_ID', 'e41ae7d9-a536-426d-b40e-f2488b11bf95')

print(f'API Key loaded: {"yes" if UMAMI_API_KEY else "NO — create .env from .env.example"}')
print(f'Website ID: {UMAMI_WEBSITE_ID}')

API Key loaded: yes
Website ID: e41ae7d9-a536-426d-b40e-f2488b11bf95


## 1. Connect to Umami Cloud API

Umami Cloud base URL: `https://api.umami.is/v1`  
Auth: `x-umami-api-key` header  
Rate limit: 50 calls / 15 seconds

In [2]:
from agent.umami_client import UmamiClient, ms_timestamp

umami = UmamiClient(api_key=UMAMI_API_KEY, website_id=UMAMI_WEBSITE_ID)

# Test connection: get active users
active = umami.get_active()
print(f'Active users right now: {active}')

Active users right now: {'visitors': 0}


## 2. Fetch Website Stats (last 7 days)

In [3]:
start = ms_timestamp(days_ago=7)
end = ms_timestamp(days_ago=0)

stats = umami.get_stats(start_at=start, end_at=end)
print('Website stats (7 days):')
for k, v in stats.items():
    if k != 'comparison':
        print(f'  {k}: {v}')

Website stats (7 days):
  pageviews: 27
  visitors: 24
  visits: 25
  bounces: 24
  totaltime: 179


## 3. Top Pages

In [4]:
top_pages = umami.get_metrics(start_at=start, end_at=end, metric_type='path', limit=15)
print('Top pages:')
for page in top_pages:
    print(f'  {page["y"]:>5} visitors — {page["x"]}')

Top pages:
      4 visitors — /amo/12/
      3 visitors — /
      3 visitors — /x402/
      3 visitors — /blog/25/#allocating-risk-across-asset-classes
      2 visitors — /quantum/amo/6/
      2 visitors — /blog/25/
      2 visitors — /quantum/amo/
      1 visitors — /blog/
      1 visitors — /blog/7/
      1 visitors — /de/
      1 visitors — /blog/13/
      1 visitors — /quantum/qml/
      1 visitors — /blog/11/
      1 visitors — /de/quantum/hardware/


## 4. Top Referrers

In [5]:
referrers = umami.get_metrics(start_at=start, end_at=end, metric_type='referrer', limit=15)
print('Top referrers:')
for ref in referrers:
    print(f'  {ref["y"]:>5} visitors — {ref["x"]}')

Top referrers:
      1 visitors — github.com
      1 visitors — fretchen.eu


## 5. Events (tracked funnels)

In [6]:
events = umami.get_metrics(start_at=start, end_at=end, metric_type='event', limit=30)
print('Tracked events:')
for event in events:
    print(f'  {event["y"]:>5}x — {event["x"]}')

Tracked events:
      3x — wallet-dropdown-open
      2x — wallet-dropdown-close


## 6. Pageviews Over Time

In [7]:
pageviews = umami.get_pageviews(start_at=start, end_at=end, unit='day')
print('Daily pageviews:')
for pv in pageviews.get('pageviews', []):
    print(f'  {pv["x"][:10]} — {pv["y"]} views')

print('\nDaily sessions:')
for s in pageviews.get('sessions', []):
    print(f'  {s["x"][:10]} — {s["y"]} sessions')

Daily pageviews:
  2026-04-05 — 1 views
  2026-04-06 — 6 views
  2026-04-07 — 12 views
  2026-04-08 — 3 views
  2026-04-09 — 1 views
  2026-04-10 — 4 views

Daily sessions:
  2026-04-05 — 1 sessions
  2026-04-06 — 6 sessions
  2026-04-07 — 9 sessions
  2026-04-08 — 3 sessions
  2026-04-09 — 1 sessions
  2026-04-10 — 4 sessions


## 7. Parse into Insights Model

Combine all data into our `Insights` Pydantic model.

In [8]:
from datetime import datetime
from agent.models import Insights, WebsiteAnalytics, EventFunnel

# Group events by funnel prefix
funnels = {}
for event in events:
    name = event['x']
    count = event['y']
    # Extract funnel prefix (e.g., 'wallet' from 'wallet-connect-attempt')
    prefix = name.split('-')[0] if '-' in name else name
    if prefix not in funnels:
        funnels[prefix] = EventFunnel()
    if 'hover' in name:
        funnels[prefix].hovers += count
    elif 'click' in name or 'attempt' in name:
        funnels[prefix].clicks += count
    elif 'success' in name:
        funnels[prefix].successes += count

# Calculate conversion rates
for f in funnels.values():
    if f.clicks > 0:
        f.conversion = f.successes / f.clicks

insights = Insights(
    website_analytics=WebsiteAnalytics(
        pageviews=stats.get('pageviews', 0),
        visitors=stats.get('visitors', 0),
        visits=stats.get('visits', 0),
        bounces=stats.get('bounces', 0),
        totaltime=stats.get('totaltime', 0),
        top_pages=[{'path': p['x'], 'visitors': p['y']} for p in top_pages[:10]],
        top_referrers=[{'referrer': r['x'], 'visitors': r['y']} for r in referrers[:10]],
        top_events=[{'event': e['x'], 'count': e['y']} for e in events[:10]],
        event_funnels=funnels,
    ),
    last_analysis=datetime.now(),
)

print(f'Insights parsed: {insights.website_analytics.visitors} visitors, '
      f'{len(funnels)} funnels detected')
print(f'Funnels: {list(funnels.keys())}')

Insights parsed: 24 visitors, 1 funnels detected
Funnels: ['wallet']


## 8. Save to Local State

In [9]:
from agent.storage import LocalStorage

store = LocalStorage(base_dir='../state')
store.write('insights.json', insights)

# Verify round-trip
loaded = store.read('insights.json')
print(f'Saved and loaded insights.json ({len(str(loaded))} chars)')
print(f'Top page: {loaded["website_analytics"]["top_pages"][0] if loaded["website_analytics"]["top_pages"] else "none"}')

Saved and loaded insights.json (930 chars)
Top page: {'path': '/amo/12/', 'visitors': 4}


In [10]:
umami.close()
print('Done — Umami ingestion validated.')

Done — Umami ingestion validated.
